In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm
import os
import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns = None
from IPython.display import display

In [ ]:
file_path = os.path.join("..", "data", "processed", "removed_high_corr.csv")

# Read CSV with index_col=0 to avoid unnamed column issues
data = pd.read_csv(file_path, index_col=0)
data

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedKFold

# Define features (X) and target (y)
X = data.drop(columns=['Exit_Status'])  # Drop target column
y = data['Exit_Status']  # Target variable

# Initialize Random Forest model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# Define RFE with Cross-Validation (10 folds)
rfecv = RFECV(
    estimator=rf_model, 
    step=1, 
    cv=StratifiedKFold(n_splits=10), 
    scoring='accuracy', 
    n_jobs=-1
)

# Fit the RFE model
rfecv.fit(X, y)

# Get selected features
selected_features = X.columns[rfecv.support_]

# Print results
print(f"Optimal number of features: {rfecv.n_features_}")
print("Selected Features:")
print(selected_features)


In [ ]:
# Get selected features based on RFE support mask
selected_features = X.columns[rfecv.support_]  # Only keep selected features

# Extract feature importances for the selected features
feature_importances = pd.Series(rfecv.estimator_.feature_importances_, index=selected_features)

# Get the top 10 most important features
top_features = feature_importances.nlargest(10)

# Reverse the gradient so the top bar is darkest
num_features = len(top_features)
colors = [(0, 0.5, 0, alpha) for alpha in np.linspace(0.3, 1, num_features)]  # Darker green at the top

# Plot feature importance with reversed green gradient
plt.figure(figsize=(10, 6))
top_features.sort_values().plot(kind='barh', color=colors, edgecolor='black')

plt.xlabel("Feature Importance Score", fontsize=12, fontweight="bold")
plt.ylabel("Feature Name", fontsize=12, fontweight="bold")
plt.title("Top 10 Most Important Features", fontsize=14, fontweight="bold", pad=10)
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.show()

# Create a new DataFrame with the selected features + target variable
data_subset = data[selected_features.tolist() + ['Exit_Status']]

# Display first few rows of the subset data
print("Subset Data Ready for Prediction:")
data_subset


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd

# Define X (all features except the target variable)
X = data.drop(columns=['Exit_Status'])  # Drop target column
y = data['Exit_Status']  # Target variable

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Print shape of training and testing sets
print(f"Training data shape: {X_train.shape}, {y_train.shape}")
print(f"Testing data shape: {X_test.shape}, {y_test.shape}")


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV

# Standardization
scaler = StandardScaler()

# Feature Selection using L1 Regularization
feature_selector = SelectFromModel(LogisticRegression(penalty='l1', solver='liblinear', random_state=42))

# Optimized Logistic Regression with Hyperparameter Tuning
lr_model = LogisticRegression(max_iter=1000, multi_class='multinomial', solver='saga', random_state=42)

# Pipeline for Scaling, Feature Selection, and Model Training
pipeline = Pipeline([
    ('scaler', scaler),
    ('feature_selection', feature_selector),
    ('log_reg', lr_model)
])

# Hyperparameter tuning
param_grid = {
    'log_reg__C': [0.001, 0.01, 0.1, 1, 10]  # Regularization strength
}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Best Model
best_model = grid_search.best_estimator_

# Predictions
lr_preds = best_model.predict(X_test)

# Compute Accuracy
lr_accuracy = accuracy_score(y_test, lr_preds)

# Display performance metrics
print("Optimized Logistic Regression Model:")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Accuracy: {lr_accuracy:.4f}")

In [ ]:
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

# Define the pipeline with scaling
pipeline = Pipeline([
    ('scaler', StandardScaler()),  # Feature scaling
    ('svm', SGDClassifier(loss='hinge', random_state=42))
])

# Hyperparameter grid
param_grid = {
    'svm__alpha': [0.0001, 0.001, 0.01, 0.1],  # Regularization strength
    'svm__max_iter': [1000, 2000, 3000],  # Number of iterations
    'svm__penalty': ['l2', 'l1', 'elasticnet'],  # Regularization type
    'svm__early_stopping': [True]  # Enables early stopping
}

# Grid search with cross-validation
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Best Model
best_model = grid_search.best_estimator_

# Predictions
svm_preds = best_model.predict(X_test)

# Compute Accuracy
svm_accuracy = accuracy_score(y_test, svm_preds)

# Display performance metrics
print("Optimized Support Vector Machine (SGDClassifier) Model:")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Accuracy: {svm_accuracy:.4f}")


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

# Define Random Forest model
rf_model = RandomForestClassifier(
    oob_score=True,  # Use out-of-bag samples for validation
    n_jobs=-1,       # Enable parallel processing
    random_state=42
)

# Hyperparameter grid
param_grid = {
    'n_estimators': [100, 200, 300],  # More trees for better performance
    'max_depth': [10, 20, None],  # Try deeper trees for complex patterns
    'min_samples_split': [5, 10, 20],  # Prevent overfitting
    'min_samples_leaf': [1, 5, 10],  # Control leaf size
    'max_features': ['sqrt', 'log2', None],  # Optimize feature selection per tree
    'class_weight': ['balanced', None]  # Handle class imbalance
}

# Grid search with cross-validation
grid_search = GridSearchCV(rf_model, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Best Model
best_model = grid_search.best_estimator_

# Predictions
rf_preds = best_model.predict(X_test)

# Compute Accuracy
rf_accuracy = accuracy_score(y_test, rf_preds)

# Display performance metrics
print("Optimized Random Forest Model:")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"OOB Score: {best_model.oob_score_:.4f}")  # Out-of-bag validation score
print(f"Accuracy: {rf_accuracy:.4f}")


In [ ]:
# Store results in a DataFrame
performance_df = pd.DataFrame({
    "Model": ["Logistic Regression", "SVM (SGDClassifier)", "Random Forest"],
    "Accuracy": [lr_accuracy, svm_accuracy, rf_accuracy]
})

# Display performance metrics
print("\nModel Performance Comparison After Resampling:")
print(performance_df)


# More Performance Matrices to Evaluate models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, log_loss, classification_report
from sklearn.model_selection import GridSearchCV

# Standardization
scaler = StandardScaler()

# Feature Selection using L1 Regularization
feature_selector = SelectFromModel(LogisticRegression(penalty='l1', solver='liblinear', random_state=42))

# Optimized Logistic Regression with Hyperparameter Tuning
lr_model = LogisticRegression(max_iter=1000, multi_class='multinomial', solver='saga', random_state=42)

# Pipeline for Scaling, Feature Selection, and Model Training
pipeline = Pipeline([
    ('scaler', scaler),
    ('feature_selection', feature_selector),
    ('log_reg', lr_model)
])

# Hyperparameter tuning
param_grid = {
    'log_reg__C': [0.001, 0.01, 0.1, 1, 10]  # Regularization strength
}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Best Model
best_model = grid_search.best_estimator_

# Predictions
lr_preds = best_model.predict(X_test)
lr_probs = best_model.predict_proba(X_test)  # Get predicted probabilities for Log Loss and ROC-AUC

# Compute Metrics
lr_accuracy = accuracy_score(y_test, lr_preds)
lr_f1_score = f1_score(y_test, lr_preds, average='weighted')  # Weighted F1-score for multi-class
lr_roc_auc = roc_auc_score(y_test, lr_probs, multi_class='ovr')  # ROC-AUC for multi-class
lr_log_loss = log_loss(y_test, lr_probs)

# Display performance metrics
print("Optimized Logistic Regression Model:")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Accuracy: {lr_accuracy:.4f}")
print(f"F1-Score: {lr_f1_score:.4f}")
print(f"ROC-AUC Score: {lr_roc_auc:.4f}")
print(f"Log Loss: {lr_log_loss:.4f}")


In [ ]:
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, roc_curve
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import label_binarize

# Standardization
scaler = StandardScaler()

# Feature Selection using L1 Regularization
feature_selector = SelectFromModel(LogisticRegression(penalty='l1', solver='liblinear', random_state=42))

# Optimized Logistic Regression with Hyperparameter Tuning
lr_model = LogisticRegression(max_iter=1000, multi_class='multinomial', solver='saga', random_state=42)

# Pipeline for Scaling, Feature Selection, and Model Training
pipeline = Pipeline([
    ('scaler', scaler),
    ('feature_selection', feature_selector),
    ('log_reg', lr_model)
])

# Hyperparameter tuning
param_grid = {
    'log_reg__C': [0.001, 0.01, 0.1, 1, 10]  # Regularization strength
}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Best Model
best_model = grid_search.best_estimator_

# Predictions
lr_preds = best_model.predict(X_test)
lr_probs = best_model.predict_proba(X_test)  # Get predicted probabilities for ROC

# Compute Metrics
lr_accuracy = accuracy_score(y_test, lr_preds)
lr_f1_score = f1_score(y_test, lr_preds, average='weighted')  # Weighted F1-score for multi-class
lr_roc_auc = roc_auc_score(y_test, lr_probs, multi_class='ovr')  # ROC-AUC for multi-class

# Display performance metrics
print("Optimized Logistic Regression Model:")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Accuracy: {lr_accuracy:.4f}")
print(f"F1-Score: {lr_f1_score:.4f}")
print(f"ROC-AUC Score: {lr_roc_auc:.4f}")

# Plot ROC Curves for each class
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])  # Binarize true labels for multi-class ROC
n_classes = y_test_bin.shape[1]

# Plot ROC curve for each class
plt.figure(figsize=(10, 8))
for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], lr_probs[:, i])
    plt.plot(fpr, tpr, label=f'Class {i} (AUC = {roc_auc_score(y_test_bin[:, i], lr_probs[:, i]):.2f})')

# Plot diagonal line (random classifier)
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random Classifier')

# Customize plot
plt.title('ROC Curves for Multi-Class Classification')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='best')
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve
from sklearn.preprocessing import label_binarize

# Define the pipeline with scaling
pipeline = Pipeline([
    ('scaler', StandardScaler()),  # Feature scaling
    ('svm', SGDClassifier(loss='log_loss', random_state=42))  # Change 'hinge' to 'log_loss'
])

# Hyperparameter grid
param_grid = {
    'svm__alpha': [0.0001, 0.001, 0.01, 0.1],  # Regularization strength
    'svm__max_iter': [1000, 2000, 3000],  # Number of iterations
    'svm__penalty': ['l2', 'l1', 'elasticnet'],  # Regularization type
    'svm__early_stopping': [True]  # Enables early stopping
}

# Grid search with cross-validation
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Best Model
best_model = grid_search.best_estimator_

# Predictions
svm_preds = best_model.predict(X_test)
svm_probs = best_model.predict_proba(X_test)  # Get predicted probabilities for ROC and AUC

# Compute Metrics
svm_accuracy = accuracy_score(y_test, svm_preds)
svm_f1_score = f1_score(y_test, svm_preds, average='weighted')  # Weighted F1-score for multi-class
svm_roc_auc = roc_auc_score(y_test, svm_probs, multi_class='ovr')  # ROC-AUC for multi-class

# Display performance metrics
print("Optimized Support Vector Machine (SGDClassifier) Model:")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Accuracy: {svm_accuracy:.4f}")
print(f"F1-Score: {svm_f1_score:.4f}")
print(f"ROC-AUC Score: {svm_roc_auc:.4f}")

# Plot ROC Curves for each class
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])  # Binarize true labels for multi-class ROC
n_classes = y_test_bin.shape[1]

# Plot ROC curve for each class
plt.figure(figsize=(10, 8))
for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], svm_probs[:, i])
    plt.plot(fpr, tpr, label=f'Class {i} (AUC = {roc_auc_score(y_test_bin[:, i], svm_probs[:, i]):.2f})')

# Plot diagonal line (random classifier)
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random Classifier')

# Customize plot
plt.title('ROC Curves for Multi-Class Classification (SVM)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='best')
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve
from sklearn.preprocessing import label_binarize

# Define Random Forest model
rf_model = RandomForestClassifier(
    oob_score=True,  # Use out-of-bag samples for validation
    n_jobs=-1,       # Enable parallel processing
    random_state=42
)

# Hyperparameter grid
param_grid = {
    'n_estimators': [100, 200, 300],  # More trees for better performance
    'max_depth': [10, 20, None],  # Try deeper trees for complex patterns
    'min_samples_split': [5, 10, 20],  # Prevent overfitting
    'min_samples_leaf': [1, 5, 10],  # Control leaf size
    'max_features': ['sqrt', 'log2', None],  # Optimize feature selection per tree
    'class_weight': ['balanced', None]  # Handle class imbalance
}

# Grid search with cross-validation
grid_search = GridSearchCV(rf_model, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Best Model
best_model = grid_search.best_estimator_

# Predictions
rf_preds = best_model.predict(X_test)
rf_probs = best_model.predict_proba(X_test)  # Get predicted probabilities for ROC and AUC

# Compute Metrics
rf_accuracy = accuracy_score(y_test, rf_preds)
rf_f1_score = f1_score(y_test, rf_preds, average='weighted')  # Weighted F1-score for multi-class
rf_roc_auc = roc_auc_score(y_test, rf_probs, multi_class='ovr')  # ROC-AUC for multi-class

# Display performance metrics
print("Optimized Random Forest Model:")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"OOB Score: {best_model.oob_score_:.4f}")  # Out-of-bag validation score
print(f"Accuracy: {rf_accuracy:.4f}")
print(f"F1-Score: {rf_f1_score:.4f}")
print(f"ROC-AUC Score: {rf_roc_auc:.4f}")

# Plot ROC Curves for each class
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])  # Binarize true labels for multi-class ROC
n_classes = y_test_bin.shape[1]

# Plot ROC curve for each class
plt.figure(figsize=(10, 8))
for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], rf_probs[:, i])
    plt.plot(fpr, tpr, label=f'Class {i} (AUC = {roc_auc_score(y_test_bin[:, i], rf_probs[:, i]):.2f})')

# Plot diagonal line (random classifier)
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random Classifier')

# Customize plot
plt.title('ROC Curves for Multi-Class Classification (Random Forest)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='best')
plt.grid(True)
plt.show()
